# 香港特别行政区河道, 海洋监测站坐标提取

点位坐标原始数据来自香港特区环境部门地表水质量年报, 为PDF文件中的表格. 

* 数据年报为PDF文件, 从中提取包含监测站坐标的页面, 将其中的监测点坐标文本复制粘贴至另行新建的文本文档中, 编写程序包对内容进行解析. 
* 河道监测站: 
    * 2015年 (含) 至2019年 (含) 的水质年报中, 仅显示监测点位在特区的分布地图, 未提供精确的地理坐标信息; 
    * 2020年 (含) 至2024年 (含) 的水质年报中, 包含监测点在特区的分布地图和详细的坐标 (地理基准: `WGS-84`). 
* 海洋监测站: 2015年 (含) 至2024年 (含) 的水质年报中包含监测点的详细坐标 (地理基准: `WGS-84`)

In [1]:
import io, sys, os; nb_dir = os.getcwd(); 
sys.path.append(nb_dir); 

In [2]:
import sqlite3; 

In [3]:
import re; 
import collections as coll, itertools as it; 

In [4]:
import numpy as np; 

In [5]:
_ = sys.stdout; 
with io.BytesIO() as sys.stdout: import idlpy; 
sys.stdout = _; 

## 指定文本文件的名称和所在路径

### 文件名匹配
文件名称格式: \
河流监测站: `Pg页码_riverreport监测年度四位数码.txt`\
海洋监测站: `Pg页码_marinereport监测年度四位数码.txt`

据此构建正则表达式, 在相对路径的目录下搜索并匹配符合条件的文件名. 

In [6]:
file_riv_filter, file_mar_filter= tuple(re.compile(
    r"Pg[0-9]+_{aq}report(?P<year>[0-9]{{4}}).txt".format(aq=aq)
) for aq in ("river", "marine")); 

In [7]:
wq_riv_spot_filename_txt = dict(); 
wq_mar_spot_filename_txt = dict(); 
for cur_dir, sub_dirs, sub_files in os.walk(nb_dir, topdown=True): 
    for file in sub_files: 
        match = file_riv_filter.match(file); 
        if match: 
            wq_riv_spot_filename_txt.update(
                {match.groupdict()["year"]: match.group(0)}
            ); 
        match = file_mar_filter.match(file); 
        if match: 
            wq_mar_spot_filename_txt.update(
                {match.groupdict()["year"]: match.group(0)}
            ); 
    break; 

In [8]:
wq_riv_spot_coord_txt = list((yr, os.sep.join(
        [nb_dir, wq_riv_spot_filename_txt[str(yr)]]
    ).format(year=yr)) for yr in range(2020, 2025)
); 
wq_mar_spot_coord_txt = list((yr, os.sep.join(
        [nb_dir, wq_mar_spot_filename_txt[str(yr)]]
    ).format(year=yr)) for yr in range(2015, 2025)
); 

## 文本文件的打开与读取

由于pdf页面内表格的文字, 在直接复制粘贴至文本时出现排版错乱, 因此需要手动调整排版:

In [9]:
def utf8_readline_gen(strm): 
    #注意: utf-8编码字符流的前三个字节(一个字符)用于表示编码的字节顺序, 不用于存储信息
    strm.seek(3); 
    while True: 
        line = strm.readline(); 
        if line == str(): 
            break; 
        yield line; 

### 河流水质监测站点坐标读取与解析
河流水质监测站点坐标格式: 
```
Shing Mun River
KY1 22° 21' 39.8" N 114° 12' 32.0" E
TR17 22° 23' 47.5" N 114° 11' 41.4" E
...
Lam Tsuen River
TR12 22° 27' 01.9" N 114° 09' 27.9" E
TR12B 22° 27' 41.0" N 114° 08' 49.6" E
...
```

In [10]:
wq_spot_riv_rec_filter = re.compile(
    u"\x20".join( [
    ur"(?P<site>[A-Z]+[0-9]+[A-Z]?)", 
    ur"(?P<lat_d>[0-9]+)°", ur"(?P<lat_m>[0-9]+)'",
    ur"(?P<lat_s>[0-9]+(\.[0-9]+)?)\"", ur"N", 
    ur"(?P<lon_d>[0-9]+)°", ur"(?P<lon_m>[0-9]+)'", 
    ur"(?P<lon_s>[0-9]+(\.[0-9]+)?)\"", ur"E"
    ] ), re.UNICODE
)

In [11]:
def wq_spot_riv_rec_parse(rec): 
    match = wq_spot_riv_rec_filter.match(rec); 
    if match: 
        res_grp = match.groupdict(); 
        res_lat = float(res_grp["lat_d"]) + float(res_grp["lat_m"]) / 60 + \
            float(res_grp["lat_s"]) / 3600; 
        res_lon = float(res_grp["lon_d"]) + float(res_grp["lon_m"]) / 60 + \
            float(res_grp["lon_s"]) / 3600; 
        res = {"site": res_grp["site"], "lat": res_lat, "lon": res_lon}; 
    else: 
        res = dict(); 
    return res; 

In [12]:
def wq_spot_riv_coord(filename, year): 
    res = list(); 
    with io.open(filename, mode="r", encoding="utf-8") as strm: 
        for line in utf8_readline_gen(strm): 
            rec_parse = wq_spot_riv_rec_parse(line); 
            if rec_parse: 
                res.append( [
                    rec_parse["site"], year, 
                    rec_parse["lon"], rec_parse["lat"]
                ] ); 
    return res; 

In [13]:
wq_spot_riv_coords = list(it.chain(
    *(wq_spot_riv_coord(file, yr) for yr, file in wq_riv_spot_coord_txt)
) ); 

### 海洋水质监测站点坐标读取与解析
海洋水质监测站点坐标格式: 
```
TM2 22° 24.744' N 114° 13.085' E 4
TM3 TS3 22° 26.857' N 114° 12.181' E 7
...
* TT1 * TS7 22° 27.270' N 114° 12.717' E 6
...
```

In [14]:
wq_spot_mar_rec_filter = re.compile(
    u"\x20".join( [
    ur"(\*|\* )?(?P<site>[A-Z]+[0-9]+)" + \
    ur" ((\*|\* )?(?P<site_sediment>[A-Z]+[0-9]+))?", 
    ur"(?P<lat_d>[0-9]+)°", 
    ur"(?P<lat_m>[0-9]+(\.[0-9]+)?)\'", ur"N", 
    ur"(?P<lon_d>[0-9]+)°", 
    ur"(?P<lon_m>[0-9]+(\.[0-9]+)?)\'", ur"E", 
    ur"(?P<depth>[0-9]+)"
    ] ), re.UNICODE
)

In [15]:
def wq_spot_mar_rec_parse(rec): 
    match = wq_spot_mar_rec_filter.match(rec); 
    if match: 
        res_grp = match.groupdict(); 
        res_lat = float(res_grp["lat_d"]) + float(res_grp["lat_m"]) / 60; 
        res_lon = float(res_grp["lon_d"]) + float(res_grp["lon_m"]) / 60; 
        res_depth = float(res_grp["depth"]); 
        res = {
            "site": res_grp["site"], "lat": res_lat, 
            "lon": res_lon, "depth": res_depth
        }; 
    else: 
        res = dict(); 
    return res; 

In [16]:
def wq_spot_mar_coord(filename, year): 
    res = list(); 
    with io.open(filename, mode="r", encoding="utf-8") as strm: 
        for line in utf8_readline_gen(strm): 
            rec_parse = wq_spot_mar_rec_parse(line); 
            if rec_parse: 
                res.append( [
                    rec_parse["site"],  year, 
                    rec_parse["lon"], rec_parse["lat"], rec_parse["depth"]
                ] ); 
    return res; 

In [17]:
wq_spot_mar_coords = list(it.chain(
    *(wq_spot_mar_coord(file, yr) for yr, file in wq_mar_spot_coord_txt)
) ); 

## 投影变换

将所有监测站点坐标从`WGS-84`地理坐标系, 经`UTM`变换, 投影至6°分带第49带的直角坐标系 \
(`UTM Zone 49N`, EPSG: 32649)

### 通过`idlpy`构造投影变换任务

In [18]:
idlpy.IDL.e = idlpy.IDL.envi(headless=True); 

% Restored file: ENVI.
% Compiled module: ENVI_VECTOR_MASK_RASTER_CLASSIC.
% Loaded DLM: XML.


In [19]:
idl_proj_utm_49n = idlpy.IDL.ENVIStandardRasterSpatialRef(
    coord_sys_code=32649, projcs=True, 
    pixel_size=np.array([0., 0.]), 
    tie_point_pixel=np.array([0., 0.]), 
    tie_point_map=np.array([0., 0.])
)

% Loaded DLM: MAP_PE.


In [20]:
idl_reproj_task = idlpy.IDL.ENVITask('ConvertGeographicToMapCoordinates'); 
idl_reproj_task.spatial_reference = idl_proj_utm_49n; 

In [21]:
for rec in wq_spot_riv_coords: 
    idl_reproj_task.input_coordinate = np.array(rec[2: 4]); 
    idl_reproj_task.execute(); 
    rec.extend(idl_reproj_task.output_coordinate); 
for rec in wq_spot_mar_coords: 
    idl_reproj_task.input_coordinate = np.array(rec[2: 4]); 
    idl_reproj_task.execute(); 
    rec.extend(idl_reproj_task.output_coordinate); 

## 提取结果建表入库

### 字段声明与二维表建立
在数据库中建立两个二维表, 分别存储河流和海洋水质监测数据, 其字段名称采用前述[字段重命名](#字段重命名)的结果. 

#### 字段名称与类型声明

In [22]:
wq_riv_fields_norm = ("spot", "year", "lon", "lat", "x", "y"); 
wq_mar_fields_norm = ("spot", "year", "lon", "lat", "depth", "x", "y"); 

In [23]:
wq_riv_field_descr_iter = zip(
    wq_riv_fields_norm, ("TEXT", "INTEGER") + ("REAL", ) * 4
); 
wq_riv_field_descr_sql = ",\x20".join(
    "{fld}\x20{cls}".format(
        fld=fld, cls=cls
    ) for (fld, cls) in wq_riv_field_descr_iter
); 
wq_riv_field_param_sql = "({0})".format(
    ",\x20".join("?" for _ in wq_riv_fields_norm)
); 

wq_mar_field_descr_iter = zip(
    wq_mar_fields_norm, ("TEXT", "INTEGER") + ("REAL", ) * 5
); 
wq_mar_field_descr_sql = ",\x20".join(
    "{fld}\x20{cls}".format(
        fld=fld, cls=cls
    ) for (fld, cls) in wq_mar_field_descr_iter
); 
wq_mar_field_param_sql = "({0})".format(
    ",\x20".join("?" for _ in wq_mar_fields_norm)
); 

print wq_riv_field_descr_sql
print wq_riv_field_param_sql
print wq_mar_field_descr_sql
print wq_mar_field_param_sql

spot TEXT, year INTEGER, lon REAL, lat REAL, x REAL, y REAL
(?, ?, ?, ?, ?, ?)
spot TEXT, year INTEGER, lon REAL, lat REAL, depth REAL, x REAL, y REAL
(?, ?, ?, ?, ?, ?, ?)


#### 建立数据库连接并创建表格

In [24]:
wq_items_sqlite = sqlite3.connect(os.sep.join(
    [nb_dir, "wq_spot_value.sqlite"]
) ); 

In [25]:
wq_items_sqlite.execute(
    """
    DROP TABLE If EXISTS info_spot_coord_river; 
    """
); 
wq_items_sqlite.execute(
    """
    CREATE TABLE info_spot_coord_river (
        {field_descr}
    ); 
    """.format(field_descr=wq_riv_field_descr_sql)
); 
wq_items_sqlite.commit(); 

In [26]:
wq_items_sqlite.execute(
    """
    DROP TABLE If EXISTS info_spot_coord_marine; 
    """
); 
wq_items_sqlite.execute(
    """
    CREATE TABLE info_spot_coord_marine (
        {field_descr}
    ); 
    """.format(field_descr=wq_mar_field_descr_sql)
); 
wq_items_sqlite.commit(); 

### 数据的插入

In [27]:
wq_items_sqlite.executemany(
    """
    INSERT Into info_spot_coord_river
        VALUES {param}
    """.format(param=wq_riv_field_param_sql), 
    wq_spot_riv_coords
); 
wq_items_sqlite.executemany(
    """
    INSERT Into info_spot_coord_marine
        VALUES {param}
    """.format(param=wq_mar_field_param_sql), 
    wq_spot_mar_coords
); 
wq_items_sqlite.commit(); 

### 数据库保存与关闭

In [28]:
wq_items_sqlite.execute("VACUUM"); 
wq_items_sqlite.commit(); 
wq_items_sqlite.close(); 